MultiModal RAG

In [3]:
import fitz #Pymupdf
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np
import torch
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

c:\Users\Vansh Parmar\Documents\Generative_AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#load the CLIP model
from dotenv import load_dotenv
load_dotenv()

#set up environment
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

#intilize the CLIP model for unified embedding
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 73139.37it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05, eleme

In [5]:
model=init_chat_model("groq:meta-llama/llama-4-scout-17b-16e-instruct")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018BE38674A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018BBF3F3D40>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
#Embedding functions
def embed_image(image_data):
    """Embed image using CLIP"""
    if isinstance(image_data,str): #if image path is given(URL)
        image=Image.open(image_data).convert("RGB")
    else: #if image_data is image
        image=image_data
    
    inputs=clip_processor(images=image,return_tensors="pt")

    with torch.no_grad():
        #takes embeddings 
        features=clip_model.get_image_features(**inputs)
        features=features.pooler_output
        #Normalize embeddings to unit vector
        features=features/features.norm(dim=-1,keepdim=True)
    return features.squeeze().numpy()

def embed_text(text):
    """Embed text using CLIP"""
    inputs=clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77   #CLIP's max token length
    )
    with torch.no_grad():
        features=clip_model.get_text_features(**inputs)
        features=features.pooler_output
        #print(features)
        #Normalize embeddings
        features=features/features.norm(dim=-1,keepdim=True)
        return features.squeeze().numpy()
        



In [31]:
#process pdf
pdf_path="multimodal_sample.pdf"
#pdf_path="attention-is-all-you-need.pdf"
docs=fitz.open(pdf_path)

#storage for all docs and embeddings
all_docs=[]
all_embeddings=[]
image_data_store={} #store actual image data for llm

#text splitter
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)

In [32]:
docs

Document('multimodal_sample.pdf')

In [33]:
for i,page in enumerate(docs):
    #process text
    text=page.get_text()
    if text.strip():
        #create temporary document for splitting
        temp_doc=Document(page_content=text, metadata={"page":i,"type":"text"})
        text_chunks=splitter.split_documents([temp_doc])

        #embed each chuk using CLIP
        for chunk in text_chunks:
            embedding=embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)

    #process image

    #1. convert PDF image to PIL format
    #2. store as base64 for LLM
    #3. create CLIP embeddingfor retrieval

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref=img[0]
            base_img=docs.extract_image(xref)
            image_bytes=base_img["image"]

            #convert to PIL image
            pil_img=Image.open(io.BytesIO(image_bytes)).convert("RGB")

            #create unique identifier
            image_id=f"page_{i}_img_{img_index}"

            #store image as base64 for later use with LLM
            buffered=io.BytesIO()
            pil_img.save(buffered,format="PNG")
            img_base64=base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id]=img_base64

            #embed image using CLIP
            embedding=embed_image(pil_img)
            all_embeddings.append(embedding)

            #create document for image
            img_doc=Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page":i, "type":"image","image_id":image_id}
            )
            all_docs.append(img_doc)
        
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

docs.close()







In [34]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
 Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]

In [35]:
#create unified FAISS vector store with CLIP embeddings
embedding_array=np.array(all_embeddings)
embedding_array

array([[-0.00267243,  0.01283001, -0.0518314 , ..., -0.00385087,
         0.0297772 , -0.00010684],
       [ 0.01732337, -0.0132769 , -0.02427029, ...,  0.0899405 ,
        -0.00272157,  0.03253039]], shape=(2, 512), dtype=float32)

In [36]:
(all_docs,all_embeddings)

([Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
  Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')],
 [array([-2.67242640e-03,  1.28300143e-02, -5.18314019e-02,  4.14879508e-02,
         -2.33942028e-02, -7.55863870e-03, -3.67659144e-02,  1.19710609e-01,
          8.52081254e-02,  2.05426686e-03, -1.11534875e-02, -1.29592679e-02,
          5.25014587e-02, -3.65392724e-03,  4.76078689e-02,  1.58372782e-02,
          2.03388035e-02,  4.35362123e-02, -3.29168420e-03,  2.03181989e-02,
          1.88025588e-03, -4.23493832e-02,  5.44102304e-03,  3

In [37]:


#create custom FAISS index since we have precomputed embeddings
vector_store=FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embedding_array)],
    embedding=None,  #we using precomputed embeddings
    metadatas=[doc.metadata for doc in all_docs]
)
vector_store

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [38]:
def retrive_model(query,k=5):
    """Unified retrieval using CLIP embedding for both text and images"""
    #embed query using CLIP
    query_embedding=embed_text(query)

    #search in unified vector store
    results=vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    return results

In [39]:
def create_multimodal_message(query,retrieved_docs):
    """create a message with both text and images for LLM"""
    content=[]

    #add the query
    content.append({
        "type" : "text",
        "text": f"Question: {query}\n\nContext:\n"
    })

    #separate text and image documents
    text_docs=[doc for doc in retrieved_docs if doc.metadata.get("type")=="text"]
    image_docs=[doc for doc in retrieved_docs if doc.metadata.get("type")=="image"]

    #add text content
    if text_docs:
        text_context="\n\n".join([
            f"[Page {doc.metadata['page']}]; {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type":"text",
            "text":f"text excepts:\n{text_context}\n"
        })

    #Add images
    for doc in image_docs:
        image_id=doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type":"text",
                "text":f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type":"image_url",
                "image_url":{
                    "url":f"data:imae/png;base64,{image_data_store[image_id]}"
                }
            })
    #Add instruction
    content.append({
        "type":"text",
        "text":"\n\nPlease answer the question based on the provided text and images."
    })
    return HumanMessage(content=content)

In [40]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal rag."""

    #Retrive relevant documents
    context_docs=retrive_model(query,k=5)

    #create multimodal message
    message=create_multimodal_message(query,context_docs)

    #get response from LLM
    response=model.invoke([message])

    #print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type=doc.metadata.get("type","unknown")
        page=doc.metadata.get("page","?")
        if doc_type=="text":
            preview=doc.page_content[:100] + "..." if len(doc.page_content)>100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"   - Image from page {page}")
    print("\n")

    return response.content

In [42]:
if __name__ == "__main__":
    queries=[
        "What does the chart on page 0 show about revenue trends?",
        "summarize the main findings from the document",
        "What visual element are present in the document?"
    ]
    # queries=[
    #    "How many attention heads does the Transformer use, and what is the dimension of each head? "
    # ]

    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer=multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("*" * 70)


Query: What does the chart on page 0 show about revenue trends?
--------------------------------------------------

Retrieved 2 documents:
  - Text from page 0: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...
   - Image from page 0


Answer: The chart on page 0 shows that revenue grew steadily across Q1, Q2, and Q3. The blue bar represents Q1, the green bar represents Q2, and the red bar represents Q3. The chart indicates the following revenue trends:

*   Q1 had a moderate level of revenue, which is represented by the shortest blue bar.
*   Q2 had a higher level of revenue than Q1, as indicated by the taller green bar. This suggests that marketing campaigns were effective in driving growth.
*   Q3 had the highest level of revenue, as shown by the tallest red bar. This indicates that global expansion had a significant impact on revenue growth.

Overall, the chart shows a steady increase in revenue across the three quarters, with 